In [ ]:
import pdfplumber
from collections import defaultdict
with pdfplumber.open("./电商行业财税合规与实务指南.pdf") as pdf:
    text = ""
    p = 1
    page_content = defaultdict(str)
    for page in pdf.pages:
        text += f"page:{p}\n\n" + page.extract_text() + "\n" + "—"*50 + "\n"
        page_content[p] += page.extract_text()
        if p != 288:
            p += 1
    print(text)

In [ ]:
page_content

In [ ]:
menu = [i for i in text.split("\n") if "..........." in i]

In [ ]:
title_page = {menu[i].split("..")[0] : (int(re.sub(r'\D', '', menu[i][-10:])), int(re.sub(r'\D', '', menu[i+1][-10:]))) for i in range(len(menu)-1)}
title_page['8.3.5分销商发票管理案例分析\u200b '] = (288, 288)

In [ ]:
prompt = f"""# Role
你是一台独立运行的「行业知识提炼引擎」，专精于从单一的书籍段落中**向上抽象**出具备普适性、可复用性的行业标准 KNOW-HOW。你的输出将直接进入并发的知识库构建流水线，与其他抽取结果进行最终归并。

# Objective
基于输入的书籍文本片段，**独立执行一次完整的知识泛化与提炼**，生成结构化的 Markdown 格式 KNOW-HOW 片段。
- **核心任务**：剥离文本中的冗余叙述、具体案例细节或背景铺垫，提炼出适用于该类业务场景的通用规范、底层准则、操作SOP或核心理论。
- **去重策略**：无需关心书籍的其他章节或其他并发节点，专注于将当前段落提炼到最精炼、最泛化的形态，便于后续的自动去重与合并。

# Compression (质量优化指令)
在生成本次 KNOW-HOW 前，必须执行以下质量优化：

1. **升维抽象**：绝对禁止简单缩写或大段摘抄原文。必须向上抽象一层，将书中的“具体案例/举例”提炼为“**通用业务准则/原理**”。
   - ❌ 差示例（就事论事）："书中提到A公司因为没有保存XML文件导致被罚款，所以要保存XML格式。"
   - ✅ 优示例（升维抽象）："电子会计凭证无纸化归档的通用准则：归档文件必须包含完整数字签名的原始文件（如XML），违规将导致税务合规风险。"

2. **结构化重组**：使用 Markdown 标题层级清晰定义知识边界。必须包含核心逻辑，并视原文内容有条件地提取适用范畴、豁免/例外、政策/理论依据。

3. **原子化剥离**：确保本次抽取的 KNOW-HOW 是一个**独立的、自洽的知识单元**，读者即使没有读过这本书的前后文，也能完全看懂并应用该知识点。

# Strict Constraints (抽取红线)
1. **绝对泛化原则**：`Know_How` 内容必须剔除所有特定案例数据（如张三、A企业、特定年份金额等），仅保留可复用的业务逻辑和专业框架。
2. **原文依从性**：所有的规则、机制和理论必须源自 `<Book_Paragraph>` 中明确提及的内容，**严禁脑补**原文未涉及的额外政策或知识。
3. **格式纯净度**：输出内容必须是可直接入库的知识片段，不含任何解释性前言（如"这段文字讲述了..."）或结语。
4. **零重复/无价值过滤**：如果 `<Book_Paragraph>` 仅为毫无专业价值的背景故事、名人名言、目录前言，或纯粹的感情抒发，无法提炼出专业规则，则判定为"无通用知识可抽取"，输出空字符串。

# Input Data
<Book_Paragraph>
{}
</Book_Paragraph>

# Workflow (执行步骤)
请在脑海中完成以下推演，最终输出要求的 JSON：

1. **文本定性**：分析 `<Book_Paragraph>` —— 这段内容是纯叙事/举例/废话？还是包含了实质性的专业知识、业务规则、判断逻辑或合规标准？
   - 如果是前者，标记为"无泛化价值"，准备输出空。
   - 如果是后者，进入步骤2。

2. **要素解构**：提取文本中的核心逻辑要素：
   - 核心机制/规则（这段话到底在教导什么专业知识？）
   - 适用条件（在什么场景或前提下生效？）
   - 风险/影响（如果不这么做有什么后果？）
   - 依据/出处（是否有提及具体的法规、准则或理论名称？）

3. **泛化重构**：将具体的描述转化为行业通用表述，用专家的口吻进行重写。

4. **Markdown结构化**：使用专业排版生成最终片段。

# Output Format
严格按照以下 JSON 格式输出。不要包含任何额外的 Markdown 标记（如 ```json）。
**⚠️ 转义要求**：将 Markdown 文本中的所有换行符替换为 `\n`，双引号替换为 `\"`，确保合法 JSON。

{{
  "Logic_Diagnosis": "说明本次提炼逻辑。例如：'该段落通过具体企业案例讲解了研发费用加计扣除的风险，已泛化为通用税务合规准则' 或 '该段落仅为作者个人经历的背景叙述，无通用业务规则可抽取'。",
  "Know_How": "### [Know-How概括性标题] \n- **核心逻辑解析**：请基于 `<Book_Paragraph>` 提炼核心专业知识。若涉及财税/业务实操，需拟人化呈现专家视角的深度剖析：精准点明该知识点的实质与核心关切。若涉及会计处理，围绕业务实质展示应用逻辑；若涉及税务处理，按合规口径推导最优处理与风险防控方案；若为模糊/新兴场景，结合行业惯例剖析处理方式的影响。要求自然连贯、环环相扣、深度解构（而非简单复述原文），字数控制在 200-400 字以内，展现真实的专业推演路径。\n- **适用范畴**：若原文有明确界定或可合理推导的适用场景，则需抽取。\n- **政策/理论依据**：若原文提及了具体法规、文件文号、会计准则或经典理论，则需抽取；若无则忽略此项。" 
}}
// 注意：当 Logic_Diagnosis 判定为"无泛化价值"时，"Know_How" 的值必须严格为空字符串 \"\"
"""

In [ ]:
# 电商行业财税合规与实务指南know-how抽取
from tqdm import tqdm
title_kh = {}
for k,v in tqdm(title_page.items()):
    p_list = list(range(v[0], v[1]+1))
    core_content = [page_content[c] for c in p_list]
    whole = "## 内容标题:" + "\n\n" + k + "\n\n" + "## 具体内容:" + "\n\n" + "\n\n".join(core_content)
    title_kh[k] = chat(prompt.format(whole))
    # print(prompt.format(whole))

In [ ]:
import concurrent.futures
from tqdm import tqdm
import time

# 1. 定义单个任务的执行函数 (Worker)
def extract_single_know_how(title, page_range, page_content_dict, prompt_template, max_retries=3):
    """
    处理单个标题的提取任务，包含重试机制
    """
    p_list = list(range(page_range[0], page_range[1] + 1))
    
    # 防止因为缺页报错，使用 dict.get() 更安全
    core_content = [page_content_dict.get(c, "") for c in p_list]
    
    # 拼装文本
    whole_text = f"## 内容标题:\n\n{title}\n\n## 具体内容:\n\n" + "\n\n".join(core_content)
    
    # 格式化 Prompt (注意：根据上文的prompt，这里变量名应该是 paragraph=whole_text)
    formatted_prompt = prompt_template.format(whole_text)
    
    # 带有重试机制的 API 调用
    for attempt in range(max_retries):
        try:
            # 调用你的 LLM 函数
            result = chat(formatted_prompt) 
            return title, result
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt) # 指数退避重试 (1s, 2s, 4s...)
            else:
                print(f"\n[Error] 提取失败: '{title}', 错误信息: {e}")
                return title, None # 失败返回 None

# 2. 定义多线程调度的主函数
def process_all_multithreaded(title_page_dict, page_content_dict, prompt_template, max_workers=5):
    """
    多线程调度函数
    :param max_workers: 最大并发线程数。请根据你 LLM API 的并发限制（QPS/RPM）进行调整。
    """
    results_kh = {}
    
    # 使用 ThreadPoolExecutor
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        # 提交所有任务到线程池
        # future_to_title 是一个字典，映射 Future 对象到对应的 title，方便出错时追踪
        future_to_title = {
            executor.submit(
                extract_single_know_how, 
                title=k, 
                page_range=v, 
                page_content_dict=page_content_dict, 
                prompt_template=prompt_template
            ): k
            for k, v in title_page_dict.items()
        }
        
        # 使用 tqdm 结合 as_completed 动态显示进度
        # as_completed 会在某个线程完成时立即 yielded
        for future in tqdm(concurrent.futures.as_completed(future_to_title), 
                           total=len(future_to_title), 
                           desc="Extracting Know-How"):
            title, result = future.result()
            
            # 只有成功提取的才存入字典（或者你可以存入特殊标记以便后续重跑）
            if result is not None:
                results_kh[title] = result
                
    return results_kh

# ================= 使用示例 =================

# 设置并发数，注意：如果用的是 OpenAI/阿里/智谱等 API，请留意账号的并发限额
# 免费/低配 API 建议设为 3-5；高配 API 可以设为 10-20
WORKERS = 16

print(f"启动多线程抽取，并发数: {WORKERS}...")
title_kh = process_all_multithreaded(
    title_page_dict=title_page,
    page_content_dict=page_content,
    prompt_template=prompt,
    max_workers=WORKERS
)

print(f"抽取完成！共成功处理 {len(title_kh)} 条数据。")

In [ ]:
with open("ecommerce_orientation_know_how.json", "w", encoding="utf-8") as f:
    json.dump(title_kh, f, indent=4, ensure_ascii=False)